In [4]:
import json
from datasets import load_dataset

from evalforge.utils import pprint

## Load the dataset

In [1]:
# https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

DATASET_NAME = "Amazon-Reviews-2023"
CATEGORY = "Clothing_Shoes_and_Jewelry" # beware, this is huge!


In [5]:
def load_category(category):
    dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", 
                       f"raw_review_{category}", split="full", trust_remote_code=True)
    dataset_meta = load_dataset("McAuley-Lab/Amazon-Reviews-2023", 
                            f"raw_meta_{category}", split="full", trust_remote_code=True)
    print(f"Loaded {len(dataset)} reviews and {len(dataset_meta)} metadata")
    pprint(dataset[0])
    print("-"*100)
    pprint(dataset_meta[0])
    return dataset, dataset_meta

In [6]:
dataset, dataset_meta = load_category(CATEGORY)

## Merge on items and reviews

- `parent_asin` is the ASIN of the product
- `title_meta` is the title of the product
- `title_review` is the title of the review

In [38]:
import pandas as pd

# Convert reviews and metadata datasets to pandas DataFrames
reviews_df = pd.DataFrame(dataset)
metadata_df = pd.DataFrame(dataset_meta)

# Perform the merge operation on 'parent_asin'
merged_df = pd.merge(reviews_df, 
                     metadata_df, 
                     left_on="parent_asin", 
                     right_on="parent_asin", 
                     suffixes=("_review", "_meta"))

# # Select relevant columns
# merged_df = merged_df[['parent_asin', 'title_meta', "title_review", 
#                       'description', 'rating', 'text', 'helpful_vote', 
#                       'verified_purchase','timestamp']]

# Display merged DataFrame
print(merged_df.head())

  parent_asin                                         title_meta  \
0  B00YQ6X8EO  Herbivore - Natural Sea Mist Texturizing Salt ...   
1  B081TJ8YS3  All Natural Vegan Dry Shampoo Powder - Eco Fri...   
2  B097R46CSY  New Road Beauty - Creamsicle - Variety 3 Pack ...   
3  B09JS339BZ  muaowig Ombre Body Wave Bundles 1B Grey Human ...   
4  B08BZ63GMJ  Yinhua Electric Nail Drill Kit Portable Profes...   

                                title_review  \
0  Such a lovely scent but not overpowering.   
1     Works great but smells a little weird.   
2                                       Yes!   
3                          Synthetic feeling   
4                                         A+   

                                         description  rating  \
0  [If given the choice, weÕd leave most telltale...     5.0   
1                                                 []     4.0   
2  [New Road Beauty Paraffin Wax is recommended f...     5.0   
3  [Hair Material: Brazilian Virgin Human Hair

In [32]:
# Specify the parent_asin of the item you're interested in
specific_asin = "B00YQ6X8EO"  # Replace this with the ASIN you're searching for

# Filter the DataFrame to retrieve all rows related to this specific ASIN
item_reviews = merged_df[merged_df["parent_asin"] == specific_asin]

# Display the filtered DataFrame
print(item_reviews)


        rating                                       title_review  \
0          5.0          Such a lovely scent but not overpowering.   
13142      1.0                                      Not worth it.   
20924      5.0                                               love   
20925      5.0                           Mermaid hair in a bottle   
20926      2.0  It makes my curly/wavy hair way too smooth and...   
...        ...                                                ...   
21024      3.0                                        Three Stars   
21025      5.0                                     Smells amazing   
21026      5.0                                     Smells AMAZING   
21028      5.0                                         Five Stars   
230293     5.0           Love the smell of coconut? Miss the sea?   

                                                     text  \
0       This spray is really nice. It smells really go...   
13142   This does not work as well as other sea m

In [36]:
item_reviews["title_meta"]

0         Herbivore - Natural Sea Mist Texturizing Salt ...
13142     Herbivore - Natural Sea Mist Texturizing Salt ...
20924     Herbivore - Natural Sea Mist Texturizing Salt ...
20925     Herbivore - Natural Sea Mist Texturizing Salt ...
20926     Herbivore - Natural Sea Mist Texturizing Salt ...
                                ...                        
21024     Herbivore - Natural Sea Mist Texturizing Salt ...
21025     Herbivore - Natural Sea Mist Texturizing Salt ...
21026     Herbivore - Natural Sea Mist Texturizing Salt ...
21028     Herbivore - Natural Sea Mist Texturizing Salt ...
230293    Herbivore - Natural Sea Mist Texturizing Salt ...
Name: title_meta, Length: 101, dtype: object

In [ ]:
# product_task = """
# You are an expert copywriter. You need to write an e-
# commerce product description based on the product
# details and customer reviews. Your description
# should be SEO-optimized. It should use an active
# voice and include the product's features,
# benefits, unique selling points without
# overpromising, and a call to action for the buyer
# • Benefits describe how product features will
# work for the buyer, addressing exactly how the
# product will improve their lives. Clearly
# distinguish between features (e.g., lightweight,
# USB-chargeable) and benefits (e.g., convenience,
# nutritious drinks on-the-go). Don't mention
# weaknesses of the product or use generic or
# repetitive language. Don't make up review text or
# quotes. Don't include any links. Don't cite the
# reviews too heavily. Divide your description into
# readable chunks divided by relevant subheadings.
# Keep your description around 200 words, no more
# than 300, in Markdown format.
# {document}
# """
# product_metric_details = """
# absence of negative reviews, absence of links, adherence to markdown format, and word count limitation, with only the first criterion requiring LLM implementation
# """
# product_dataset_url = ""